# Neural Machine Translation ENG to SPA con Encoder-Decoder + Atención + Pre-trained Embeddings (SpaCy)

In [1]:
# ==============================================================================
# EJERCICIO ACADÉMICO: MACHINE TRANSLATION (EN -> ES)
# Arquitectura: Encoder-Decoder + Atención + Pre-trained Embeddings (SpaCy)
# ==============================================================================

# 1. IMPORTACIÓN DE LIBRERÍAS Y CONEXIÓN A DRIVE
import os
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, LSTM, Dense, Embedding, Attention, Concatenate, TimeDistributed
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
import spacy


2026-05-02 09:39:04.138890: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2026-05-02 09:39:04.179327: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2026-05-02 09:39:04.206296: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1777732744.246214   10080 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1777732744.255723   10080 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-05-02 09:39:04.326187: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU ins

# Dataset ENG-SPA

In [2]:
import re

# --- NUEVA CARGA DE DATOS (ManyThings) ---
!wget http://www.manythings.org/anki/spa-eng.zip
!unzip -o spa-eng.zip

def preprocess_sentence(s):
    # Limpieza básica: minúsculas y quitar signos de puntuación
    s = s.lower().strip()
    s = re.sub(r"([?.!,¿])", r" \1 ", s) # Separar signos de las palabras
    s = re.sub(r'[" "]+', " ", s)
    s = re.sub(r"[^a-zA-Z?.!,¿áéíóúÁÉÍÓÚñÑ]+", " ", s)
    return s.strip()

def load_data(path, num_samples=60000):
    en_sentences = []
    es_sentences = []
    with open(path, 'r', encoding='utf-8') as f:
        lines = f.read().split('\n')

    for line in lines[:num_samples]:
        parts = line.split('\t')
        if len(parts) >= 2:
            en_sentences.append(preprocess_sentence(parts[0]))
            # El target (español) lleva los tokens de control
            es_sentences.append('<start> ' + preprocess_sentence(parts[1]) + ' <end>')
    return en_sentences, es_sentences

# Cargamos 60,000 ejemplos para un equilibrio entre calidad y velocidad
english_texts, spanish_texts = load_data('spa.txt', num_samples=60000)

print(f"Muestra del dataset: {english_texts[100]} -> {spanish_texts[100]}")

--2026-05-02 09:39:08--  http://www.manythings.org/anki/spa-eng.zip
Resolving www.manythings.org (www.manythings.org)... 173.254.30.110
Connecting to www.manythings.org (www.manythings.org)|173.254.30.110|:80... connected.
HTTP request sent, awaiting response... 200 OK
Length: 5502374 (5.2M) [application/zip]
Saving to: ‘spa-eng.zip.2’

spa-eng.zip.2       100%[===================>]   5.25M  5.47MB/s    in 1.0s    

2026-05-02 09:39:09 (5.47 MB/s) - ‘spa-eng.zip.2’ saved [5502374/5502374]

Archive:  spa-eng.zip
  inflating: _about.txt              
  inflating: spa.txt                 
Muestra del dataset: go now . -> <start> vayan ahora mismo . <end>


In [3]:
# Configuración de rutas
SAVE_PATH = './model/'
if not os.path.exists(SAVE_PATH):
    os.makedirs(SAVE_PATH)

# ==============================================================================
# 2. CARGA DE DATOS Y PREPROCESAMIENTO
# ==============================================================================

# Añadir tokens especiales al target (Español)
spanish_texts = ['<start> ' + s + ' <end>' for s in spanish_texts]

# Tokenización
en_tokenizer = Tokenizer(filters='')
en_tokenizer.fit_on_texts(english_texts)
en_seq = en_tokenizer.texts_to_sequences(english_texts)

es_tokenizer = Tokenizer(filters='')
es_tokenizer.fit_on_texts(spanish_texts)
es_seq = es_tokenizer.texts_to_sequences(spanish_texts)

# Calculamos las longitudes máximas reales del dataset cargado
max_len_en = max(len(s) for s in en_seq)
max_len_es = max(len(s) for s in es_seq)

encoder_input_data = pad_sequences(en_seq, maxlen=max_len_en, padding='post')
decoder_input_data = pad_sequences(es_seq, maxlen=max_len_es, padding='post')

# Crear el target desplazado
decoder_target_data = np.zeros_like(decoder_input_data)
decoder_target_data[:, :-1] = decoder_input_data[:, 1:]

# Vocabularios
en_vocab_size = len(en_tokenizer.word_index) + 1
es_vocab_size = len(es_tokenizer.word_index) + 1



In [5]:
# ==============================================================================
# 3. EMBEDDINGS PREENTRENADOS CON SPACY
# ==============================================================================
# Cargamos modelos de spacy para extraer los vectores (usaremos md para tener vectores reales)
!python -m spacy download en_core_web_md
!python -m spacy download es_core_news_md

nlp_en = spacy.load("en_core_web_md")
nlp_es = spacy.load("es_core_news_md")

def get_embedding_matrix(tokenizer, nlp_model, vocab_size, dim=300):
    matrix = np.zeros((vocab_size, dim))
    for word, i in tokenizer.word_index.items():
        if i < vocab_size:
            matrix[i] = nlp_model(word).vector
    return matrix

en_embedding_matrix = get_embedding_matrix(en_tokenizer, nlp_en, en_vocab_size)
es_embedding_matrix = get_embedding_matrix(es_tokenizer, nlp_es, es_vocab_size)


[notice] A new release of pip is available: 26.0.1 -> 26.1
[notice] To update, run: pip install --upgrade pip
ERROR: Operation cancelled by user
^C
  Using cached https://github.com/explosion/spacy-models/releases/download/es_core_news_md-3.8.0/es_core_news_md-3.8.0-py3-none-any.whl (42.3 MB)

[notice] A new release of pip is available: 26.0.1 -> 26.1
[notice] To update, run: pip install --upgrade pip
✔ Download and installation successful
You can now load the package via spacy.load('es_core_news_md')


In [ ]:
# ==============================================================================
# 4. DEFINICIÓN DEL MODELO (ENCODER - DECODER + ATTENTION)
# ==============================================================================
latent_dim = 300 # Dimensión de las unidades LSTM

# --- ENCODER ---
encoder_inputs = Input(shape=(max_len_en,), name="Enc_Input")
enc_emb = Embedding(en_vocab_size, 300, weights=[en_embedding_matrix], trainable=False)(encoder_inputs)
encoder_lstm = LSTM(latent_dim, return_sequences=True, return_state=True, name="Enc_LSTM")
encoder_outputs, state_h, state_c = encoder_lstm(enc_emb)
encoder_states = [state_h, state_c]

# --- DECODER ---
decoder_inputs = Input(shape=(max_len_es,), name="Dec_Input")
dec_emb_layer = Embedding(es_vocab_size, 300, weights=[es_embedding_matrix], trainable=False)
dec_emb = dec_emb_layer(decoder_inputs)

decoder_lstm = LSTM(latent_dim, return_sequences=True, return_state=True, name="Dec_LSTM")
decoder_outputs, _, _ = decoder_lstm(dec_emb, initial_state=encoder_states)

# --- MECANISMO DE ATENCIÓN (Luong Style) ---
# Comparamos la salida del decoder con todas las salidas del encoder
attention_layer = Attention(name="Attention_Layer")
attn_out = attention_layer([decoder_outputs, encoder_outputs])

# Concatenamos el contexto de atención con la salida del decoder
decoder_concat_input = Concatenate(axis=-1, name="Concat_Layer")([decoder_outputs, attn_out])

# Capa densa final
decoder_dense = TimeDistributed(Dense(es_vocab_size, activation='softmax'))
decoder_outputs = decoder_dense(decoder_concat_input)

# Definir modelo de entrenamiento
model = Model([encoder_inputs, decoder_inputs], decoder_outputs)
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
model.summary()

# ==============================================================================
# 5. ENTRENAMIENTO Y PERSISTENCIA
# ==============================================================================
# batch ajustado para procesar más rápido en GPU
# 20-30 épocas suelen ser suficientes para ver resultados coherentes

print("\nIniciando entrenamiento...")
model.fit([encoder_input_data, decoder_input_data],
          decoder_target_data,
          batch_size=128,
          epochs=50,
          validation_split=0.1)

# Guardar el modelo completo
model.save(os.path.join(SAVE_PATH, 'nmt_model.h5'))
print(f"Modelo guardado en: {SAVE_PATH}")

import pickle

# Guardar tokenizers
with open(os.path.join(SAVE_PATH, 'tokenizer_en.pkl'), 'wb') as f:
    pickle.dump(en_tokenizer, f)

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ Enc_Input           │ (None, 10)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Dec_Input           │ (None, 18)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_2         │ (None, 10, 300)   │  2,190,000 │ Enc_Input[0][0]   │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_3         │ (None, 18, 300)   │  4,495,800 │ Dec_Input[0][0]   │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Enc_LSTM (LSTM)     │ [(None, 10, 300), │    721,200 │ embedding_2[0][0] │
│                     │ (None, 300),      │            │                   │
│                     │ (None, 300)]      │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Dec_LSTM (LSTM)     │ [(None, 18, 300), │    721,200 │ embedding_3[0][0… │
│                     │ (None, 300),      │            │ Enc_LSTM[0][1],   │
│                     │ (None, 300)]      │            │ Enc_LSTM[0][2]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Attention_Layer     │ (None, 18, 300)   │          0 │ Dec_LSTM[0][0],   │
│ (Attention)         │                   │            │ Enc_LSTM[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Concat_Layer        │ (None, 18, 600)   │          0 │ Dec_LSTM[0][0],   │
│ (Concatenate)       │                   │            │ Attention_Layer[… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ time_distributed_1  │ (None, 18, 14986) │  9,006,586 │ Concat_Layer[0][… │
│ (TimeDistributed)   │                   │            │                   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 17,134,786 (65.36 MB)

 Trainable params: 10,448,986 (39.86 MB)

 Non-trainable params: 6,685,800 (25.50 MB)


Iniciando entrenamiento...
Epoch 1/5
422/422 ━━━━━━━━━━━━━━━━━━━━ 314s 739ms/step - accuracy: 0.7883 - loss: 1.5153 - val_accuracy: 0.7633 - val_loss: 1.4938
Epoch 2/5
422/422 ━━━━━━━━━━━━━━━━━━━━ 412s 976ms/step - accuracy: 0.8336 - loss: 0.9870 - val_accuracy: 0.7911 - val_loss: 1.2784
Epoch 3/5
422/422 ━━━━━━━━━━━━━━━━━━━━ 403s 956ms/step - accuracy: 0.8517 - loss: 0.8008 - val_accuracy: 0.8052 - val_loss: 1.1647
Epoch 4/5
422/422 ━━━━━━━━━━━━━━━━━━━━ 275s 652ms/step - accuracy: 0.8660 - loss: 0.6677 - val_accuracy: 0.8151 - val_loss: 1.0898
Epoch 5/5
422/422 ━━━━━━━━━━━━━━━━━━━━ 307s 727ms/step - accuracy: 0.8789 - loss: 0.5634 - val_accuracy: 0.8225 - val_loss: 1.0367


Modelo guardado en: ./model/


# Inferencia

In [ ]:
# 1. IMPORTACIÓN DE LIBRERÍAS Y CONEXIÓN A DRIVE
import os
import numpy as np
import tensorflow as tf
import spacy

In [ ]:
from tensorflow.keras.models import load_model
import pickle

SAVE_PATH = './model/'

# Cargar el modelo desde la ruta de Drive definida anteriormente
model = load_model(os.path.join(SAVE_PATH, 'nmt_model.h5'))

# Cargar tokenizers (en una sesión nueva)
with open(os.path.join(SAVE_PATH, 'tokenizer_en.pkl'), 'rb') as f:
    en_tokenizer = pickle.load(f)

In [ ]:
# ==============================================================================
# 6. INFERENCIA (TRADUCCIÓN DE NUEVOS TEXTOS)
# ==============================================================================

def translate(input_sentence):
    # Preprocesar entrada
    input_seq = en_tokenizer.texts_to_sequences([input_sentence])
    input_seq = pad_sequences(input_seq, maxlen=max_len_en, padding='post')

    # El estado inicial viene del encoder
    decoded_sentence = '<start>'

    for i in range(max_len_es):
        target_seq = es_tokenizer.texts_to_sequences([decoded_sentence])
        target_seq = pad_sequences(target_seq, maxlen=max_len_es, padding='post')

        # Predecir siguiente token
        output_tokens = model.predict([input_seq, target_seq], verbose=0)

        # Tomar el índice con mayor probabilidad para el paso actual
        # En inferencia paso a paso, nos interesa la palabra en la posición 'i'
        sampled_token_index = np.argmax(output_tokens[0, i, :])
        sampled_char = es_tokenizer.index_word.get(sampled_token_index, '')

        if sampled_char == '<end>' or sampled_char == '':
            break

        decoded_sentence += ' ' + sampled_char

    return decoded_sentence.replace('<start>', '').strip()

In [ ]:
# --- CELDA DE PRUEBA ---
texto_a_traducir = "the house is big"
#texto_a_traducir = "The red car"
#texto_a_traducir = "The engineer did it"

resultado = translate(texto_a_traducir)
print(f"\nEnglish: {texto_a_traducir}")
print(f"Spanish: {resultado}")


English: the house is big
Spanish: el dinero es grande .


# Taller para clase

1. Consulte acerca de la base de datos cnn_dailymail de Hugging Face. Qué problema le permitiría resolver un modelo que se entrene con esta base de datos?

2. Cargue la base de datos en el notebook desde Hugging Face.

3. Entrene su propio arquitectura Encoder-Decoder + Atención + pretrained embeddings para resolver esta tarea y muestre que tal funciona.
